# 08 — Your SKILL.md Is Just a Text File. Ours Executes, Measures, and Improves.

**The pitch.** Claude Code's AGENTS.md and Cursor's `.cursor/rules` are static text — the AI reads them and tries to follow. No parameter validation. No quality scoring before execution. No feedback loop. When the skill degrades, you find out from a bad output, not a metric. mycontext's `SKILL.md` is an **executable artifact**: typed parameters, pattern fusion, quality-gated execution, and a continuous improvement pipeline from run logs.

**Who this is for:** Teams building reusable AI agent behaviors, anyone currently using AGENTS.md or .cursor/rules who wants programmatic control.

**The four differentiators:**
1. **Typed parameters** — `input_schema` in frontmatter defines and validates params before execution
2. **Pattern fusion** — `pattern: risk_assessor` fuses skill body with a cognitive template's reasoning structure
3. **Quality-gated execution** — `SkillRunner` scores the assembled prompt and blocks if quality drops below threshold
4. **Continuous improvement** — `skill_health_report` + `suggested_edits_from_log` track quality over runs and suggest fixes

When a skill uses **pattern fusion**, the underlying free templates often ship **default `self_check` questions** on `Constraints` — the same quality story as Prompt Architect and `QualityMetrics` (0.11+; use **≥ 0.11.1** so fused contexts include quality paragraphs under **GUARD RAILS** in `assemble()`).

**Next:** See [README.md](README.md) for the full series index.

## Research grounding

> **Typed parameters:** Parameter validation before execution prevents silent failures — the same class of bugs as untyped Python with plausible-looking output. Static skill files (AGENTS.md, .cursor/rules) have no validation layer; mycontext's `input_schema` adds one.

> **Quality gating:** Pre-execution structural scoring catches prompt defects before token cost is incurred *(Zhou et al. 2022 — prompt quality as pre-execution signal)*.

> **Improvement loop:** The `skill_health_report` + `suggested_edits_from_log` pipeline follows the *Reflexion* paradigm *(Shinn et al. 2023)* — systematic error analysis from execution outputs, fed back as targeted edits to the skill definition, producing measurable quality improvement over successive runs without human intervention.

## Claude/Cursor vs mycontext SKILL.md

| Feature | Claude AGENTS.md | Cursor .cursor/rules | mycontext SKILL.md |
|---------|-----------------|---------------------|--------------------|
| Parameters | Free text | Free text | Typed schema, validated |
| Reasoning structure | None | None | Pattern fusion (88 cognitive patterns) |
| Pre-execution quality check | None | None | `QualityMetrics` gating |
| Execution feedback loop | None | None | `skill_health_report` from logs |
| Improvement pipeline | Manual edit | Manual edit | `improve_skill_with_llm` |
| Format | Markdown | Markdown | Markdown + YAML frontmatter |

## Install and setup

In [1]:
# %pip install -q -U "mycontext-ai>=0.11.1"
import os
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from dotenv import load_dotenv
from IPython.display import display_markdown

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

def md(s):
    display_markdown(s, raw=True)

import mycontext

md(f"**mycontext-ai** `{getattr(mycontext, '__version__', '?')}`")


**mycontext-ai** `0.11.0`

## Step 1 — Write an executable SKILL.md

Three formats, progressively more powerful:

In [2]:
from pathlib import Path

skills_dir = Path("_demo_skills")
skills_dir.mkdir(exist_ok=True)

# Skill A: basic (like Claude/Cursor — plain text, no schema)
(skills_dir / "basic_skill").mkdir(exist_ok=True)
(skills_dir / "basic_skill" / "SKILL.md").write_text("""\
---
name: code_security_check
description: Review code for common security vulnerabilities
---

Review the provided code for security issues.
Focus on injection, authentication, and data exposure risks.
""")
md("Basic skill written (no schema, no pattern — same as Claude/Cursor)")


Basic skill written (no schema, no pattern — same as Claude/Cursor)

In [3]:
# Skill B: with typed parameters (differentiator 1)
(skills_dir / "typed_skill").mkdir(exist_ok=True)
(skills_dir / "typed_skill" / "SKILL.md").write_text("""\
---
name: code_security_check_v2
description: Review code for security vulnerabilities with configurable depth
input_schema:
  language: str
  depth: str
  focus_area: str
---

Review the following {language} code with {depth} depth.
Focus specifically on: {focus_area}.

For each vulnerability found:
1. Name the vulnerability type (OWASP category)
2. Show the vulnerable line(s)
3. Explain the risk
4. Provide a concrete fix
""")
md("Typed skill written — parameters are validated before execution")


Typed skill written — parameters are validated before execution

In [4]:
# Skill C: with pattern fusion (differentiator 2)
(skills_dir / "fused_skill").mkdir(exist_ok=True)
(skills_dir / "fused_skill" / "SKILL.md").write_text("""\
---
name: code_security_check_v3
description: Security code review fused with root_cause_analyzer reasoning structure
input_schema:
  language: str
  depth: str
  focus_area: str
pattern: root_cause_analyzer
---

Review the following {language} code with {depth} depth.
Focus specifically on: {focus_area}.

Apply systematic root cause analysis to each vulnerability found:
identify the immediate cause, contributing factors, and root cause.
""")
md("Pattern-fused skill written — root_cause_analyzer reasoning baked in")


Pattern-fused skill written — root_cause_analyzer reasoning baked in

## Step 2 — Load and compare

In [5]:
from mycontext import Skill

basic = Skill.load(skills_dir / "basic_skill")
typed = Skill.load(skills_dir / "typed_skill")
fused = Skill.load(skills_dir / "fused_skill")

md(f"Basic: name={basic.name}, schema={basic.input_schema}, pattern={basic.pattern}")
md(f"Typed: name={typed.name}, schema={typed.input_schema}, pattern={typed.pattern}")
md(f"Fused: name={fused.name}, schema={fused.input_schema}, pattern={fused.pattern}")


Basic: name=code_security_check, schema={}, pattern=None

Typed: name=code_security_check_v2, schema={'language': <class 'str'>, 'depth': <class 'str'>, 'focus_area': <class 'str'>}, pattern=None

Fused: name=code_security_check_v3, schema={'language': <class 'str'>, 'depth': <class 'str'>, 'focus_area': <class 'str'>}, pattern=root_cause_analyzer

## Step 3 — `SkillRunner`: quality-gated execution (differentiator 3)

In [ ]:
from mycontext import SkillRunner
from mycontext.intelligence import QualityMetrics

runner = SkillRunner()

params = {
    "language": "Python",
    "depth": "thorough",
    "focus_area": "SQL injection and authentication",
}

# Run all three skills and compare quality scores
qm = QualityMetrics(mode="heuristic")

md(f"{'Skill':<30} {'Quality':>10} {'Gate (≥0.65)'}")
md("-" * 60)

# run() expects a Path to the skill dir (or SKILL.md); it loads internally.
for skill_name, skill_path, skill in [
    ("basic", skills_dir / "basic_skill", basic),
    ("typed", skills_dir / "typed_skill", typed),
    ("fused", skills_dir / "fused_skill", fused),
]:
    result = runner.run(
        skill_path,
        task="Review this code for vulnerabilities",
        execute=False,  # don't call LLM yet
        **{k: v for k, v in params.items() if k in skill.input_schema},
    )
    score = result.quality_score.overall if result.quality_score else 0
    gate = "✓ PASS" if score >= 0.65 else "✗ BLOCK"
    md(f"{skill_name + ' skill':<30} {score:>9.2f}   {gate}")


In [ ]:
# Quality threshold only affects execution when execute=True (see Step 4).
result_gated = runner.run(
    skills_dir / "fused_skill",
    task="Review this code for vulnerabilities",
    execute=False,
    quality_threshold=0.7,
    **params,
)

md(f"Quality score: {result_gated.quality_score.overall:.2f}")
md(f"Gated (blocked): {result_gated.gated}")
if result_gated.gated:
    md("Execution blocked — quality below threshold")
else:
    md("Quality passed — execution would proceed")


## Step 4 — With execution (differentiator 3, live)

In [ ]:
vulnerable_code = """
def get_user(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    return db.execute(query)

def login(username, password):
    if username == "admin" and password == "admin123":
        session['user'] = username
"""

result_live = runner.run(
    skills_dir / "fused_skill",
    task=f"Review this code:\n{vulnerable_code}",
    execute=True,
    provider="openai",
    quality_threshold=0.6,
    **params,
)
md(f"Quality score: {result_live.quality_score.overall:.2f}")
md(f"Gated: {result_live.gated}")
_out = result_live.execution_result
if _out is not None:
    md("\n=== Review output ===")
    _text = getattr(_out, "response", None) or str(_out)
    md((_text[:800-3] + '...' if len(_text) > 800 else _text))


## Step 5 — Improvement pipeline (differentiator 4)

In [ ]:
from mycontext.skills.improvement import improvement_report

# improvement_report gives you actionable feedback from a single run result
result_for_report = runner.run(
    skills_dir / "typed_skill",
    task="Review code for SQL injection",
    execute=False,
    **params,
)

report = improvement_report(result_for_report)
md("=== Improvement report ===")
md(report)


In [ ]:
# Log runs for health tracking
import tempfile
from pathlib import Path

log_path = Path(tempfile.mktemp(suffix=".jsonl"))
logged_runner = SkillRunner(log_runs=True, log_path=log_path)

# Simulate multiple runs
for i in range(3):
    logged_runner.run(
        skills_dir / "fused_skill",
        task=f"Run {i+1}: check code for injection vulnerabilities",
        execute=False,
        **params,
    )

md(f"Logged {sum(1 for _ in log_path.open())} runs to {log_path}")


In [ ]:
from mycontext.skills.improvement import skill_health_report

health = skill_health_report(log_path=log_path)
md("=== Skill health report ===")
md(health)


## Summary: the four differentiators

| Differentiator | Claude/Cursor | mycontext SKILL.md | API |
|---------------|---------------|--------------------|---------|
| Typed parameters | ✗ free text | ✓ `input_schema` validated | `Skill.validate_params` |
| Pattern fusion | ✗ none | ✓ `pattern:` fuses template | `SkillRunner` with `skill.pattern` |
| Quality gating | ✗ none | ✓ block if score below threshold | `SkillRunner(quality_threshold=0.7)` |
| Improvement loop | ✗ manual | ✓ logs + health report + edits | `skill_health_report`, `improvement_report` |

**You've completed the series.** See [README.md](README.md) for the full index and how to run all notebooks.